# Необходимые настройки

In [1]:
import os, subprocess
if "JAVA_HOME" not in os.environ:
    try:
        jh = subprocess.check_output(["/usr/libexec/java_home","-v","17"]).decode().strip()
        os.environ["JAVA_HOME"] = jh
    except Exception:
        pass

In [2]:
import findspark, os
findspark.init()

In [3]:
# + скролл
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

# как в пандас
# spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
# spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 50) 

In [4]:
OUT_PATH      = "./synth_big_table_parquet"  # куда писать parquet

## Библиотеки и запуск спарк-сессии

In [88]:
from pyspark.sql import SparkSession, functions as F, types as T, Window
from IPython.display import display
import gc
import pandas as pd
import numpy as np

In [7]:
try:
    spark
    # если сессию до этого останавливали — создадим заново
    _ = spark.sparkContext  # триггер обращения
except Exception:
    spark = (SparkSession.builder
             .appName("SynthBigTable")
             .master("local[*]")
             .config("spark.sql.shuffle.partitions", "24")
             .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/24 14:46:08 WARN Utils: Your hostname, Mac-mini-Artem.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.25 instead (on interface en0)
25/08/24 14:46:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/24 14:46:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
def spark_alive(spark):
    try:
        return not spark.sparkContext._jsc.sc().isStopped()
    except Exception:
        return False

spark_alive(spark)  # True = ок

True

# Генерация тренировочных значений

In [24]:
# 1) Параметры генерации
N_ROWS        = 500_000   # 1e6, 5e6, ...)
N_DAYS        = 30        
N_PARTITIONS  = 8         # параллелизм при генерации/записи
SEED          = 42        # воспроизводимость
OUT_PATH      = "./synth_big_table_parquet"  # куда писать parquet

In [25]:
# Тюним шафлы под локальную машину
spark.conf.set("spark.sql.shuffle.partitions", max(4, N_PARTITIONS*3))

In [26]:
# 2) Базовая "рамка" с нужным количеством строк
base = spark.range(0, N_ROWS, numPartitions=N_PARTITIONS)

In [27]:
# Хелпер для равномерного выбора категории без UDF
def pick_from(vals, rcol):
    # element_at в Spark использует 1-based индексацию
    return F.element_at(F.array(*[F.lit(v) for v in vals]),
                        (F.floor(rcol * F.lit(len(vals))) + 1).cast("int"))


In [28]:
# Несколько независимых источников случайных чисел
r1 = F.rand(SEED)
r2 = F.rand(SEED + 1)
r3 = F.rand(SEED + 2)
r4 = F.rand(SEED + 3)

In [29]:
df = (base
    .withColumn("cat_country", pick_from(["US","DE","FR","GB","NL","ES","IT"], r1))
    .withColumn("cat_device", F.when(r2 < 0.65, F.lit("mobile"))
                               .when(r2 < 0.90, F.lit("desktop"))
                               .otherwise(F.lit("tablet")))
    .withColumn("cat_segment", pick_from(["free","pro","enterprise"], r3))
    .withColumn("num_amount", (F.randn(SEED) * 20 + 200).cast("double"))
    .withColumn("num_score",  (r1 * 1000).cast("double"))
    .withColumn("num_ratio",  (r2 * 10 / (r3 * 10 + F.lit(1e-6))).cast("double"))
    .withColumn("flag_is_active", (r4 < 0.30).cast("boolean"))
    .withColumn("date_dt", F.date_sub(F.current_date(), (r3 * F.lit(N_DAYS)).cast("int")))
    .withColumn("event_date", F.date_add(F.col("date_dt"), (r4 * F.lit(7)).cast("int") - F.lit(3)))
    .withColumn("value_with_nans", F.when(r1 < 0.08, F.lit(float("nan")))  # ~8% NaN
                                    .otherwise(F.randn(SEED + 7).cast("double")))
)

# Обзор

### Первый взгляд

In [30]:
print("=== Schema ===")
df.printSchema()

=== Schema ===
root
 |-- id: long (nullable = false)
 |-- cat_country: string (nullable = false)
 |-- cat_device: string (nullable = false)
 |-- cat_segment: string (nullable = false)
 |-- num_amount: double (nullable = false)
 |-- num_score: double (nullable = false)
 |-- num_ratio: double (nullable = true)
 |-- flag_is_active: boolean (nullable = false)
 |-- date_dt: date (nullable = true)
 |-- event_date: date (nullable = true)
 |-- value_with_nans: double (nullable = false)



In [31]:
print("=== Sample rows ===")
df.limit(3).show()

=== Sample rows ===
+---+-----------+----------+-----------+------------------+------------------+------------------+--------------+----------+----------+--------------------+
| id|cat_country|cat_device|cat_segment|        num_amount|         num_score|         num_ratio|flag_is_active|   date_dt|event_date|     value_with_nans|
+---+-----------+----------+-----------+------------------+------------------+------------------+--------------+----------+----------+--------------------+
|  0|         NL|   desktop| enterprise|247.68958108482332|  619.189370225301|1.0040319833670577|         false|2025-08-01|2025-08-03|-0.04167221574820542|
|  1|         GB|   desktop| enterprise|203.84186808258704|509.60188424464815|0.7564333739990915|          true|2025-07-29|2025-07-27| -0.8485901886179861|
|  2|         ES|    mobile|        pro|214.67467306657315| 832.5259388871524|0.4963285222214282|         false|2025-08-09|2025-08-10|  1.2000195221995014|
+---+-----------+----------+-----------+----

In [32]:
print("=== Random sample rows (≈0.1%) ===")
# temp_fraction  = 0.001
N = 500
temp_fraction  = min(1.0, (N / df.count()) * 1.25)
df.sample(withReplacement=False, fraction=temp_fraction, seed=1).limit(N).show(3, truncate=False)

=== Random sample rows (≈0.1%) ===
+---+-----------+----------+-----------+------------------+------------------+-------------------+--------------+----------+----------+-------------------+
|id |cat_country|cat_device|cat_segment|num_amount        |num_score         |num_ratio          |flag_is_active|date_dt   |event_date|value_with_nans    |
+---+-----------+----------+-----------+------------------+------------------+-------------------+--------------+----------+----------+-------------------+
|264|GB         |desktop   |pro        |194.89123945746135|447.77202472832613|1.5302424802274404 |true          |2025-08-10|2025-08-07|0.14164681045020927|
|276|FR         |mobile    |enterprise |174.68829445210966|392.1962187544592 |0.7891469974856216 |false         |2025-08-04|2025-08-07|-0.8026936428427666|
|800|NL         |mobile    |pro        |208.6708463826192 |622.6360740720396 |0.22615197511805088|true          |2025-08-12|2025-08-10|1.61610053243343   |
+---+-----------+----------+-

### Партицирование

In [33]:
print("=== Size / Partitions ===")
row_cnt = df.count()
npart   = df.rdd.getNumPartitions()
print(f"rows={row_cnt:,}  partitions={npart}")

=== Size / Partitions ===
rows=500,000  partitions=8


### Типы данных

In [34]:
#  Список числовых и категориальных колонок
num_types = {"double","float","int","bigint","smallint","tinyint","decimal"}
num_cols  = [c for c,t in df.dtypes if any(t.startswith(nt) for nt in num_types)]
cat_cols  = [c for c,t in df.dtypes if t == "string"]
bool_cols = [c for c,t in df.dtypes if t == "boolean"]
date_cols = [c for c,t in df.dtypes if t.startswith("date")]

print("Numeric:", num_cols)
print("Categorical:", cat_cols)
print("Boolean:", bool_cols)
print("Date-like:", date_cols)

Numeric: ['id', 'num_amount', 'num_score', 'num_ratio', 'value_with_nans']
Categorical: ['cat_country', 'cat_device', 'cat_segment']
Boolean: ['flag_is_active']
Date-like: ['date_dt', 'event_date']


### Null and NaN

In [35]:
# === Nulls ===
null_exprs = [F.sum(F.col(c).isNull().cast("long")).alias(f"{c}__nulls") for c in df.columns]
print("=== Nulls ===")
df.select(*null_exprs).show()

=== Nulls ===
+---------+------------------+-----------------+------------------+-----------------+----------------+----------------+---------------------+--------------+-----------------+----------------------+
|id__nulls|cat_country__nulls|cat_device__nulls|cat_segment__nulls|num_amount__nulls|num_score__nulls|num_ratio__nulls|flag_is_active__nulls|date_dt__nulls|event_date__nulls|value_with_nans__nulls|
+---------+------------------+-----------------+------------------+-----------------+----------------+----------------+---------------------+--------------+-----------------+----------------------+
|        0|                 0|                0|                 0|                0|               0|               0|                    0|             0|                0|                     0|
+---------+------------------+-----------------+------------------+-----------------+----------------+----------------+---------------------+--------------+-----------------+--------------------

In [36]:
# === NaNs ===
nan_exprs  = [F.sum(F.isnan(c).cast("long")).alias(f"{c}__nans") for c in num_cols]
print("=== NaNs ===")
df.select(*nan_exprs).show()

=== NaNs ===
+--------+----------------+---------------+---------------+---------------------+
|id__nans|num_amount__nans|num_score__nans|num_ratio__nans|value_with_nans__nans|
+--------+----------------+---------------+---------------+---------------------+
|       0|               0|              0|              0|                40098|
+--------+----------------+---------------+---------------+---------------------+



### Базовое описание

In [84]:
vals = [r[0] for r in df.select('cat_country').na.drop().distinct().collect()] # список уникальных значений
vals

['DE', 'US', 'NL', 'GB', 'FR', 'IT', 'ES']

In [37]:
print("=== Categorical value counts ===")
# cat_cols - тут категориальные если что

def value_cnts(data, cat,n):
    temp = (df.groupby(cat)
            .count()
            .orderBy(F.desc(cat)).toPandas())
    temp['rate'] = round(temp['count']/temp['count'].sum(),2)
    display(temp.iloc[:n,:])

for i in cat_cols:
    value_cnts(df,i,100)
gc.collect()

=== Categorical value counts ===


,cat_country,count,rate
0,US,71597,0.14
1,NL,71030,0.14
2,IT,71378,0.14
3,GB,71443,0.14
4,FR,71692,0.14
5,ES,71304,0.14
6,DE,71556,0.14


,cat_device,count,rate
0,tablet,17292,0.03
1,mobile,325442,0.65
2,desktop,157266,0.31


,cat_segment,count,rate
0,pro,166716,0.33
1,free,166743,0.33
2,enterprise,166541,0.33


2892

In [21]:
print("=== Numeric describe ===")
df.select(num_cols).describe().show()

=== Numeric describe ===


25/08/24 13:31:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+--------------------+--------------------+------------------+
|summary|                id|        num_amount|           num_score|           num_ratio|   value_with_nans|
+-------+------------------+------------------+--------------------+--------------------+------------------+
|  count|            500000|            500000|              500000|              500000|            500000|
|   mean|          249999.5|199.93138063736748|   499.4030314769453|   6.849858668159375|               NaN|
| stddev|144337.71163490156| 19.97118887031128|  288.77910293077275|   635.3266370907997|               NaN|
|    min|                 0|111.01628170273861|0.001873137899788...|3.845691470047983E-6|-4.398996390455747|
|    max|            499999| 301.0898410273889|   999.9992634449936|   325562.1969099771|               NaN|
+-------+------------------+------------------+--------------------+--------------------+------------------+



In [22]:
print("=== Numeric approx quantiles ===")
# k% значений ниже этого числа
temp_quat = 'num_amount'
qant = [0.01, 0.1, 0.2, 0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.99]

pd.DataFrame(dict(zip([str(i) for i in qant],np.array(df.approxQuantile(temp_quat, qant, 0.01)).reshape(-1,1).round(2))),index = [temp_quat])

=== Numeric approx quantiles ===


,0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.99
num_amount,111.02,173.65,182.77,189.32,194.76,199.72,204.71,210.04,216.18,224.66,301.09


In [23]:
print("=== Dates range ===")
(df.select(
    F.min("date_dt").alias("min_date"),
    F.max("date_dt").alias("max_date"),
    F.countDistinct("date_dt").alias("n_partitions_by_date_dt")
).show())

=== Dates range ===
+----------+----------+-----------------------+
|  min_date|  max_date|n_partitions_by_date_dt|
+----------+----------+-----------------------+
|2025-07-26|2025-08-24|                     30|
+----------+----------+-----------------------+



In [75]:
value_cnts(df,'date_dt',3)

,date_dt,count,rate
0,2025-08-24,16618,0.03
1,2025-08-23,16737,0.03
2,2025-08-22,16543,0.03


# Чистка

### Обработка пропусков

In [103]:
df1 = df

In [104]:
# trim и пустые строки
df1 = df1.withColumn('cat_country', F.when(F.length(F.trim(F.col('cat_country')))==0,None)
                                   .otherwise(F.trim(F.col('cat_country'))))

df1 = df1.withColumn('value_with_nans', F.when(F.isnan('value_with_nans'), None)
                                   .otherwise(F.col('value_with_nans')))

In [105]:
# Nan теперь нет
df1.select(*nan_exprs).show()

+--------+----------------+---------------+---------------+---------------------+
|id__nans|num_amount__nans|num_score__nans|num_ratio__nans|value_with_nans__nans|
+--------+----------------+---------------+---------------+---------------------+
|       0|               0|              0|              0|                    0|
+--------+----------------+---------------+---------------+---------------------+



In [106]:
# === Заполняем медианой ===

In [107]:
def approx_median(df_, col):
    qs = df_.approxQuantile(col, [0.5], 0.01)
    return qs[0] if qs else None

medians = {c: approx_median(df1, c) for c in num_cols}

In [108]:
for c in num_cols:
    df1  = (df1
        .withColumn(f'{c}__was_null', F.col(c).isNull())
        .withColumn(c,F.when(F.col(c).isNull(),F.lit(medians[c])).otherwise(F.col(c))))
    
for c in cat_cols:
    df1  = (df1
        .withColumn(f'{c}__was_null', F.col(c).isNull())
        .withColumn(c,F.when(F.col(c).isNull(),F.lit('unknown')).otherwise(F.col(c))))

for c in bool_cols:
    df1  = (df1
        .withColumn(f'{c}__was_null', F.col(c).isNull())
        .withColumn(c,F.when(F.col(c).isNull(),F.lit(False)).otherwise(F.col(c))))

In [109]:
# Проверка
print("Остались ли NaN/NULL?")
nulls = [F.sum(F.col(c).isNull().cast("long")).alias(f"{c}__nulls") for c in df1.columns]
nans  = [F.sum(F.isnan(c).cast("long")).alias(f"{c}__nans") for c in num_cols]
df1.select(*(nulls + nans)).show(truncate=False)

Остались ли NaN/NULL?
+---------+------------------+-----------------+------------------+-----------------+----------------+----------------+---------------------+--------------+-----------------+----------------------+-------------------+---------------------------+--------------------------+--------------------------+--------------------------------+----------------------------+---------------------------+----------------------------+-------------------------------+--------+----------------+---------------+---------------+---------------------+
|id__nulls|cat_country__nulls|cat_device__nulls|cat_segment__nulls|num_amount__nulls|num_score__nulls|num_ratio__nulls|flag_is_active__nulls|date_dt__nulls|event_date__nulls|value_with_nans__nulls|id__was_null__nulls|num_amount__was_null__nulls|num_score__was_null__nulls|num_ratio__was_null__nulls|value_with_nans__was_null__nulls|cat_country__was_null__nulls|cat_device__was_null__nulls|cat_segment__was_null__nulls|flag_is_active__was_null__nul

In [110]:
# Чистим от лишних колонок
was_null_cols = [i for i in df1.columns if i.find('was_null')!=-1]
df1 = df1.drop(*was_null_cols)

### Обработка выбросов

In [111]:
def find_outliers(data, cols,fill_outlier = False):
    bounds = {}
    for c in cols:
        p01, p99 = data.approxQuantile(c, [0.01, 0.99], 0.01)
        bounds[c] = (p01, p99)
        
    idx = {} # собираем id выбросов
    
    for c in cols:
        lo, hi = bounds[c]
        
        data = data.withColumn(f'{c}_is_outlier',
                    F.when(F.col(c) < F.lit(lo),1)
                     .when(F.col(c) > F.lit(hi),1)
                     .otherwise(0))
        mask = ((F.col(c)<lo) | (F.col(c)>hi))
        ids_data = data.filter(mask).select(F.col('id'))
        ids = [i[0] for i in ids_data.collect()]
        idx[c] = ids
        
        if fill_outlier == True:
                data = data.withColumn(c,
                    F.when(F.col(c) < F.lit(lo),F.lit(lo))
                     .when(F.col(c) > F.lit(hi),F.lit(hi))
                     .otherwise(F.col(c)))

    outlier_cols = [c for c in data.columns if 'is_outlier' in c]
    temp  = data.select(*[F.sum(c).alias(f'{c}_sum_outliers') for c in outlier_cols])
    temp.show()
    data = data.drop(*outlier_cols)
    return data, idx
    

In [112]:
df1,idx = find_outliers(df1,num_cols, fill_outlier = False)

+--------------------------+----------------------------------+---------------------------------+---------------------------------+---------------------------------------+
|id_is_outlier_sum_outliers|num_amount_is_outlier_sum_outliers|num_score_is_outlier_sum_outliers|num_ratio_is_outlier_sum_outliers|value_with_nans_is_outlier_sum_outliers|
+--------------------------+----------------------------------+---------------------------------+---------------------------------+---------------------------------------+
|                         0|                                 0|                                0|                                0|                                      0|
+--------------------------+----------------------------------+---------------------------------+---------------------------------+---------------------------------------+



### Сохранение

In [113]:
(df1.repartition("date_dt")
    .write.mode("overwrite")
    .partitionBy("date_dt")
    .parquet(OUT_PATH))

25/08/24 14:19:02 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [9]:
#чтение
clean = spark.read.parquet(OUT_PATH)
print("Число строк в clean:", clean.count())
print("Колонки в clean:", clean.columns)

Число строк в clean: 500000
Колонки в clean: ['id', 'cat_country', 'cat_device', 'cat_segment', 'num_amount', 'num_score', 'num_ratio', 'flag_is_active', 'event_date', 'value_with_nans', 'date_dt']


# Агрегаты/Срезы

In [10]:
from_date = clean.select(F.min(F.col('date_dt'))).first()[0]
to_date = clean.select(F.max(F.col('date_dt'))).first()[0]
print(f"Диапазон дат в данных: {from_date} … {to_date}")

Диапазон дат в данных: 2025-07-26 … 2025-08-24


In [11]:
d = (clean
     .where((F.col("date_dt") >= F.lit(from_date)) & (F.col("date_dt") <= F.lit(to_date)))
     .cache())

### GroupBy + Rollup

In [181]:

# Лучше сущить - d.select (-только нужные колонки-)
(d.groupBy('date_dt','cat_country')
     .agg(
         F.count('*').alias('rows'),
         F.sum('num_amount').alias('sum_amount'),
         F.approx_count_distinct('id',rsd = 0.02).alias('users_approx'),
         #F.countDistinct('id').alias('users_exact'), # Жестко тяжелая операция
         F.avg(F.col('flag_is_active').cast('double')).alias('share_active'),
         F.expr('percentile_approx(num_amount,0.95)').alias('p95_amount')) #95% меньше 
          .orderBy(F.desc('date_dt'),F.desc('cat_country'))
).show(6)

+----------+-----------+----+------------------+------------+-------------------+------------------+
|   date_dt|cat_country|rows|        sum_amount|users_approx|       share_active|        p95_amount|
+----------+-----------+----+------------------+------------+-------------------+------------------+
|2025-08-24|         US|2439| 485989.0061641914|        2443| 0.2820828208282083|  230.378806929584|
|2025-08-24|         NL|2388| 477825.7320780251|        2320| 0.3073701842546064|232.17655314269726|
|2025-08-24|         IT|2306| 461218.2660876337|        2295|  0.308326105810928|232.20027879017593|
|2025-08-24|         GB|2343| 467541.7968103733|        2320|0.31626120358514725|232.15759455686862|
|2025-08-24|         FR|2336| 467091.9129741674|        2279| 0.2915239726027397|233.93259604776603|
|2025-08-24|         ES|2371|473287.86534298706|        2350|0.28089413749472797| 233.0827232578103|
+----------+-----------+----+------------------+------------+-------------------+----------

In [182]:
#  + Rollup с подитогами
rollup_df = (d.rollup('date_dt','cat_country')
     .agg(
         F.count('*').alias('rows'),
         F.sum('num_amount').alias('sum_amount'),
         F.approx_count_distinct('id',rsd = 0.02).alias('users_approx'),
         #F.countDistinct('id').alias('users_exact'), # Жестко тяжелая операция
         F.avg(F.col('flag_is_active').cast('double')).alias('share_active'),
         F.expr('percentile_approx(num_amount,0.95)').alias('p95_amount'),
         F.grouping_id("date_dt", "cat_country").alias("gid")) #95% меньше 
          .orderBy(F.desc('date_dt'),F.desc('cat_country'))
)
rollup_df.where(F.col('gid')==0).show()

+----------+-----------+----+------------------+------------+-------------------+------------------+---+
|   date_dt|cat_country|rows|        sum_amount|users_approx|       share_active|        p95_amount|gid|
+----------+-----------+----+------------------+------------+-------------------+------------------+---+
|2025-08-24|         US|2439| 485989.0061641914|        2443| 0.2820828208282083|  230.378806929584|  0|
|2025-08-24|         NL|2388| 477825.7320780251|        2320| 0.3073701842546064|232.17655314269726|  0|
|2025-08-24|         IT|2306| 461218.2660876337|        2295|  0.308326105810928|232.20027879017593|  0|
|2025-08-24|         GB|2343| 467541.7968103733|        2320|0.31626120358514725|232.15759455686862|  0|
|2025-08-24|         FR|2336| 467091.9129741674|        2279| 0.2915239726027397|233.93259604776603|  0|
|2025-08-24|         ES|2371|473287.86534298706|        2350|0.28089413749472797| 233.0827232578103|  0|
|2025-08-24|         DE|2435| 487543.0005911068|       

### GroupBy + Partition

In [89]:
w = Window.partitionBy('date_dt').orderBy(F.desc('sum_amount'))

In [99]:
(d.groupBy('date_dt','cat_country')
     .agg(
         F.count('*').alias('rows'),
         F.sum('num_amount').alias('sum_amount'),
         F.approx_count_distinct('id',rsd = 0.02).alias('users_approx'),
         #F.countDistinct('id').alias('users_exact'), # Жестко тяжелая операция
         F.avg(F.col('flag_is_active').cast('double')).alias('share_active'),
         F.expr('percentile_approx(num_amount,0.95)').alias('p95_amount')) #95% меньше 
        
         .withColumn('rk',F.row_number().over(w)) # + сверху колонку которая строит ранг sum_amount по каждой дате
         .where(F.col('rk')<=3) # топ 3 sum_amount
         .orderBy(F.desc('date_dt'))
).show(6)

+----------+-----------+----+------------------+------------+-------------------+------------------+---+
|   date_dt|cat_country|rows|        sum_amount|users_approx|       share_active|        p95_amount| rk|
+----------+-----------+----+------------------+------------+-------------------+------------------+---+
|2025-08-24|         DE|2435| 487543.0005911068|        2421| 0.3047227926078029|233.06392959810498|  1|
|2025-08-24|         US|2439| 485989.0061641914|        2443| 0.2820828208282083|  230.378806929584|  2|
|2025-08-24|         NL|2388| 477825.7320780251|        2320| 0.3073701842546064|232.17655314269726|  3|
|2025-08-23|         FR|2432|486245.30835086195|        2396|0.31866776315789475|233.88661885933402|  1|
|2025-08-23|         GB|2418| 483628.1637502979|        2396| 0.3110008271298594|232.87668138906702|  2|
|2025-08-23|         ES|2412|480734.20257412986|        2409|0.30140961857379767|231.61240347879803|  3|
+----------+-----------+----+------------------+-------

### GroupBy + Pivot

In [129]:
(d.groupBy("date_dt","cat_device")
     .pivot("cat_segment", ["free","pro","enterprise"]) # Раскрываетм столбец cat_segment
     .agg(F.avg("num_amount"))
     .orderBy(F.col("free").asc_nulls_last())
    
).show(6)

+----------+----------+------------------+----+----------+
|   date_dt|cat_device|              free| pro|enterprise|
+----------+----------+------------------+----+----------+
|2025-08-18|    tablet|197.89649495968015|NULL|      NULL|
|2025-08-16|    tablet|198.92228046429585|NULL|      NULL|
|2025-08-24|    tablet| 199.1331135981261|NULL|      NULL|
|2025-08-19|    tablet|199.24958819169603|NULL|      NULL|
|2025-08-21|   desktop|199.42295878284946|NULL|      NULL|
|2025-08-20|    tablet|199.57821708413806|NULL|      NULL|
+----------+----------+------------------+----+----------+
only showing top 6 rows


### GroupBy + скользящие метрики

In [183]:
day_level = (d.groupBy("date_dt")
               .agg(F.count("*").alias("rows"),
                    F.sum("num_amount").alias("sum_amount"),
                    F.avg("num_amount").alias("avg_amount")))
w7 = Window.orderBy("date_dt").rowsBetween(-6, 0)  # 7 календарных точек
rolling = (day_level
           .withColumn("sum_amount_7d", F.sum("sum_amount").over(w7))
           .withColumn("rows_7d", F.sum("rows").over(w7))
           .withColumn("avg_amount_7d", (F.col("sum_amount_7d")/F.col("rows_7d")))
           .orderBy("date_dt"))
print("=== 7-day rolling metrics (sum/avg) ===")
rolling.show(20, truncate=False)

=== 7-day rolling metrics (sum/avg) ===


25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 18:10:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/24 1

+----------+-----+------------------+------------------+--------------------+-------+------------------+
|date_dt   |rows |sum_amount        |avg_amount        |sum_amount_7d       |rows_7d|avg_amount_7d     |
+----------+-----+------------------+------------------+--------------------+-------+------------------+
|2025-07-26|16808|3366173.715914745 |200.27211541615569|3366173.715914745   |16808  |200.27211541615569|
|2025-07-27|16664|3335365.6339659686|200.15396267198562|6701539.349880714   |33472  |200.21329319672304|
|2025-07-28|16480|3294826.3955137786|199.9287861355448 |9996365.745394493   |49952  |200.11942956026772|
|2025-07-29|16624|3327681.5859134975|200.1733389024    |1.332404733130799E7 |66576  |200.13289070097318|
|2025-07-30|16615|3318215.6748169092|199.71204783731022|1.6642263006124899E7|83191  |200.04883949135   |
|2025-07-31|16630|3322767.8545786035|199.80564369083606|1.99650308607035E7  |99821  |200.00832350611097|
|2025-08-01|16550|3310509.768063353 |200.03080169567087

# JOin

In [222]:
keys = ["id", "date_dt"]  #  ключи
# Проверяем правую таблицу (должна быть 1-строка-на-ключ)
dups_right = (
    d.groupBy(*keys).count()
         .filter("count > 1")
)

n_keys_right     = d.select(*keys).distinct().count()
n_rows_right     = d.count()
print("right distinct keys:", n_keys_right, " | right rows:", n_rows_right)
# если distinct(keys) < rows → есть дубликаты по ключам

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")  # AQE включить один раз — must have

In [ ]:
# как Спарк собирается Джонить таблицы
d.join(d, keys).explain("formatted") 
# BroadcastHashJoin (лучше для big ↔ small при равенстве по ключам)
# SortMergeJoin (классика для больших равенств, со shuffle+sort)
# ShuffledHashJoin
# BroadcastNestedLoopJoin (неравенства/кросс, обычно медленно)

	•	K — число уникальных ключей джойна (по всей совокупности ключей).
	•	L, R — число строк в левой и правой таблицах.
	•	C = sc.defaultParallelism — параллелизм кластера (≈ суммарное число ядер исполнителей).
	•	S — планируемое число shuffle-партиций (обычно spark.sql.shuffle.partitions).

In [231]:
import math

In [228]:
def advise_partitions(left, right_unique, keys, target_rows_per_part=1_000_000):
    sc = left.sparkSession.sparkContext
    C = sc.defaultParallelism
    Sconf = int(left.sparkSession.conf.get("spark.sql.shuffle.partitions", "200"))

    L = left.count()
    R = right_unique.count()
    K_left  = left.select(*keys).distinct().count()
    K_right = right_unique.select(*keys).distinct().count()
    K = min(K_left, K_right)  # для 1:1 по факту одинаково

    # Хоть и грубо, но полезно:
    S0 = math.ceil((L + R) / target_rows_per_part)
    S1 = max(2*C, min(8*C, S0))  # коридор по железу
    S  = min(S1, K)              # не больше числа ключей

    print(f"rows: left={L:,}, right={R:,}")
    print(f"distinct keys: left={K_left:,}, right={K_right:,}  -> using K={K:,}")
    print(f"defaultParallelism C={C}, shuffle.partitions (conf)={Sconf}")
    print(f"proposed S0(rows)={S0}, S1(hardware)={S1}  -> S(final)={S}")
    print(f"keys per partition ≈ {K/max(1,S):.1f} (целимся ≥ 5–10)")
    return S

In [233]:
S = advise_partitions(d, d, ["id"])

rows: left=500,000, right=500,000
distinct keys: left=500,000, right=500,000  -> using K=500,000
defaultParallelism C=8, shuffle.partitions (conf)=24
proposed S0(rows)=1, S1(hardware)=16  -> S(final)=16
keys per partition ≈ 31250.0 (целимся ≥ 5–10)


In [235]:
spark.conf.set("spark.sql.adaptive.enabled", "true")      # включи AQE
spark.conf.set("spark.sql.shuffle.partitions", "16")      # под задачу

left_p  = left.repartition(16, *keys)                     # опционально
right_p = right_unique.repartition(16, *keys)             # опционально
res = left_p.join(right_p, keys, "left")

62500.0

### Broadcast малой таблицы

In [187]:
dim_country = spark.createDataFrame(
    [
        ("US","NA",0.9), ("DE","EU",0.7), ("FR","EU",0.65), ("GB","EU",0.68),
        ("NL","EU",0.6), ("ES","EU",0.55), ("IT","EU",0.58)
    ],
    "cat_country string, region string, risk_score double"
)

In [193]:
joined_df = d.join(F.broadcast(dim_country), on = 'cat_country', how = 'left')

### Large

#### Генерация

In [210]:
HOT_ID = 42
n_users = d.agg(F.count("*")).first()[0]
N_EVENTS = int(max(1.5 * n_users, 800_000))


In [211]:
events = (spark.range(0, N_EVENTS)
    .withColumn('user_id', 
                F.when(F.rand(seed = 0)<0.20, F.lit(HOT_ID))
                .otherwise((F.rand(seed = 1)*F.lit(n_users)).cast('bigint')))
    .withColumn('event_value', (F.rand(2)*10).cast('double'))
    .repartition(max(8,int(spark.sparkContext.defaultParallelism)))
)

In [220]:
events.groupby('user_id').count().orderBy(F.desc('count')).show(5)

+-------+------+
|user_id| count|
+-------+------+
|     42|159989|
| 212296|     9|
| 121902|     9|
| 209377|     8|
|   6651|     8|
+-------+------+
only showing top 5 rows


In [196]:
users = d.select("id", "cat_country", "num_amount").distinct()

In [201]:
# Кол-во пользователей и подсказки по размерам
n_users = users.agg(F.count("*")).first()[0]
print(f"users: {n_users:,}")

users: 500,000


In [23]:
# Окончание работ

In [45]:
spark.catalog.clearCache()
spark.stop()

# Допы

In [35]:
# Если логов слишком много
spark.sparkContext.setLogLevel("WARN")

In [ ]:
spark.range(0, 100) создание колонки
spark.range(5).withColumn("rnd", F.rand(0))

In [ ]:
# работа со строкой
df.filter(df['choose_col']contains("ex")) # содержится ли в колонке "ex"

In [ ]:
оконные функции глубже (ранжирование, лаги/диффы)

In [ ]:
оптимизация ввода-вывода (кол-во файлов, coalesce/repartition, ZSTD/ Snappy, валидация partition pruning)

In [ ]:
Shuffle-based Join (Sort-Merge / Shuffle Hash)

In [ ]:
Struct lit otherwice

ПАРТИШН
w = W.partitionBy("category").orderBy(F.rand(seed))
df_k = (df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") <= k)
          .drop("rn"))

In [ ]:
Учимся кэшировать раскэшировать

In [184]:
def mem_overview(spark=None, show_exec=True, show_rdd=True):
    """
    Краткий отчёт по памяти: Python, JVM(Spark), executors storage, RDD-кэши.
    Возвращает словарь с данными и показывает таблицы (если есть pandas).
    """
    import os, sys, math
    from pprint import pprint

    # -------- helpers
    def fmt_bytes(n):
        try:
            n = int(n)
        except Exception:
            return str(n)
        units = ["B","KB","MB","GB","TB"]
        i = 0
        while n >= 1024 and i < len(units)-1:
            n /= 1024.0
            i += 1
        return f"{n:.1f} {units[i]}"

    # -------- Python/JVM память
    py = {"rss": None, "vms": None}
    jvm = {"rss_total": 0, "children": []}
    try:
        import psutil
        p = psutil.Process(os.getpid())
        mi = p.memory_info()
        py["rss"], py["vms"] = mi.rss, mi.vms
        for ch in p.children(recursive=True):
            try:
                cmd = " ".join(ch.cmdline()).lower()
            except Exception:
                cmd = ch.name().lower()
            if "java" in cmd:  # процессы Spark (driver/executor)
                mi = ch.memory_info()
                jvm["children"].append({"pid": ch.pid, "rss": mi.rss, "cmd": cmd[:120]})
                jvm["rss_total"] += mi.rss
    except Exception:
        # fallback без psutil: только приблизительный RSS
        try:
            import resource
            # на macOS ru_maxrss — в байтах, на Linux — в килобайтах
            ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            py["rss"] = ru if sys.platform == "darwin" else ru * 1024
        except Exception:
            pass

    # -------- Spark части
    exec_rows, rdd_rows, conf_mem = [], [], {}
    try:
        from pyspark.sql import SparkSession
        spark = spark or SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
        sc = spark.sparkContext

        # конфиги памяти
        conf = dict(sc.getConf().getAll())
        for k in ("spark.driver.memory", "spark.executor.memory",
                  "spark.executor.instances", "spark.memory.offHeap.enabled",
                  "spark.memory.offHeap.size"):
            if k in conf:
                conf_mem[k] = conf[k]

        if show_exec:
            # Память storage у исполнителей (включая драйвер): (total, free)
            jmap = sc._jsc.sc().getExecutorMemoryStatus()  # java.util.Map
            it = jmap.entrySet().iterator()
            while it.hasNext():
                e = it.next()
                key = str(e.getKey())      # executor id / BlockManagerId
                val = e.getValue()         # scala.Tuple2(total, free)
                total = int(val._1())
                free  = int(val._2())
                used  = total - free
                exec_rows.append({"executor": key, "total": total, "used": used, "free": free})

        if show_rdd:
            # Какие RDD/кэши занимают память
            infos = sc._jsc.sc().getRDDStorageInfo()       # scala.collection.Seq[RDDInfo]
            n = infos.length()
            for i in range(n):
                info = infos.apply(i)
                name = str(info.name()) if info.name() else f"rdd_{info.id()}"
                rdd_rows.append({
                    "id": int(info.id()),
                    "name": name,
                    "mem": int(info.memSize()),
                    "disk": int(info.diskSize()),
                    "partitions": int(info.numPartitions()),
                    "storageLevel": str(info.storageLevel())
                })
    except Exception as e:
        # Spark не инициализирован — это ок
        pass

    report = {
        "python": py,
        "jvm": jvm,
        "spark_conf_memory": conf_mem,
        "executors_storage": exec_rows,
        "rdd_storage": rdd_rows,
    }

    # -------- красивый вывод (если есть pandas)
    try:
        import pandas as pd
        from IPython.display import display
        print("Python process:", f"RSS {fmt_bytes(py['rss'])}" if py["rss"] else "n/a",
              "| VMS", fmt_bytes(py["vms"]) if py["vms"] else "")
        if jvm["children"]:
            print("JVM (Spark) total RSS:", fmt_bytes(jvm["rss_total"]), f"| java procs: {len(jvm['children'])}")

        if conf_mem:
            print("Spark memory conf:", conf_mem)

        if exec_rows:
            dfe = pd.DataFrame(exec_rows)
            for c in ("total","used","free"):
                dfe[c] = dfe[c].map(fmt_bytes)
            display(dfe.sort_values("executor").reset_index(drop=True)
                    .rename(columns={"executor":"Executor/BlockManager", "total":"Total storage", "used":"Used", "free":"Free"}))

        if rdd_rows:
            dfr = pd.DataFrame(rdd_rows)
            for c in ("mem","disk"):
                dfr[c] = dfr[c].map(fmt_bytes)
            display(dfr.sort_values("mem", ascending=False).reset_index(drop=True)
                    .rename(columns={"mem":"In-memory", "disk":"On-disk"}))
    except Exception:
        # текстовый фолбэк
        pprint(report)

    return report

In [185]:
rep = mem_overview()

Python process: RSS 53.4 MB | VMS 393.0 GB
JVM (Spark) total RSS: 409.5 MB | java procs: 1


In [ ]:
Python process: RSS 214.9 MB | VMS 393.3 GB
JVM (Spark) total RSS: 488.6 MB | java procs: 1